# Voces y documentos peruanos · Parte 1: corpus y extracción

**¿Se pueden convertir documentos escaneados y entrevistas grabadas en un archivo de texto estructurado y buscable?**

En este notebook armo los dos corpus del proyecto y los paso a texto:

- **Documentos**: 25 imágenes de documentos peruanos (8 impresos, 8 fotos con sombras o inclinación y 9 manuscritos) leídas con PaddleOCR.
- **Audio**: al menos 30 minutos de español peruano (20 clips o más, uno con varios hablantes) transcritos con faster-whisper, con detección de voz y separación de hablantes.

Al final quedan tres tablas (`documentos.csv`, `transcripciones.csv` y `fuentes.csv`) con los textos ya seudonimizados.

**Entorno: kernel de Python 3.14.7**, sin instalar otras versiones de Python. Solo uso librerías que publican versión para 3.14. PaddlePaddle, el motor original de PaddleOCR, no la tiene; por eso PaddleOCR corre con su motor de PyTorch (`engine="transformers"`), que usa los mismos modelos PP-OCRv6.

## Paso 1 · Registro de la extracción

| Dato | Valor |
|---|---|
| Fecha de extracción | 2026-10-25 |
| Python | 3.14.7 |
| PaddleOCR | 3.7.0 · PP-OCRv6 medium (`lang="es"`, motor PyTorch `engine="transformers"`) |
| Transformers | 5.20.1 |
| Modelo ASR | faster-whisper 1.2.1 · modelo `large-v3-turbo` en int8 |
| PyTorch | 2.14.1 (CPU) |
| Separación de hablantes | huellas MFCC propias (NumPy/SciPy) + clustering jerárquico de Ward (scikit-learn) |
| Hardware | CPU (sin GPU) |

| Corpus | Fuente | Licencia |
|---|---|---|
| Documentos | Wikimedia Commons | La de cada archivo (CC BY, CC BY-SA o dominio público), registrada en `datos/fuentes.csv` |
| Documentos | Fotos propias sin datos personales | CC BY 4.0 |
| Audio | OpenSLR 73, *Crowdsourced high-quality Peruvian Spanish speech* (Google) | CC BY-SA 4.0 |
| Audio | Entrevistas de Wikimedia Commons elegidas en 3.1 (opcional) | La de cada archivo, registrada en `datos/fuentes.csv` |

## Instalación en el kernel actual (Python 3.14.7)

Esta celda **no instala Python**: instala las librerías en el mismo Python que ejecuta este notebook (`sys.executable`). Lo hace **una por una** y al final muestra una tabla con cuáles se instalaron y cuáles se pueden importar, para identificar rápido cualquier problema.

Ejecútala **una sola vez**. Luego **reinicia el kernel** (*Kernel → Restart Kernel*) y continúa desde la celda de importaciones.

- No se usa `paddlepaddle` (no existe para Python 3.14): PaddleOCR funciona con PyTorch.
- No se usa `pyannote.audio`: depende de `torchcodec`, que en Windows exige tener FFmpeg instalado en el sistema. La separación de hablantes se hace con funciones propias.
- `restricciones.txt` fija OpenCV 4.10.0.84 (la versión que exige PaddleOCR), `av<19` (con PyAV 19, faster-whisper no puede leer archivos de audio) y `numpy<2.4` (PaddleX no acepta NumPy 2.4).
- **PaddleOCR se instala con `uv` y una sobrescritura de PyYAML.** PaddleX (dependencia de PaddleOCR) exige exactamente `PyYAML==6.0.2`, que no tiene instalador para Python 3.14: pip intenta compilarlo y falla (`╰─> PyYAML`). `sobrescrituras.txt` reemplaza esa exigencia por `PyYAML>=6.0.3`, la primera versión con instalador para 3.14 (es solo una corrección menor, compatible con 6.0.2). pip no permite sobrescribir dependencias, por eso para este paquete se usa `uv pip install --override`, sobre el mismo Python del kernel.

In [5]:
import subprocess
import sys
from pathlib import Path

print("Python del kernel:", sys.version.split()[0], "→", sys.executable)
if sys.version_info[:2] != (3, 14):
    print("⚠ Este notebook está preparado para un kernel de Python 3.14.")

Path("restricciones.txt").write_text(
    "opencv-python==4.10.0.84\nopencv-python-headless==4.10.0.84\nopencv-contrib-python==4.10.0.84\nav<19\nnumpy<2.4\n")
# PaddleX exige PyYAML==6.0.2, que no tiene instalador para Python 3.14 → se sobrescribe por 6.0.3 (sí lo tiene)
Path("sobrescrituras.txt").write_text("PyYAML>=6.0.3\n")
CON_SOBRESCRITURA = {"paddleocr>=3.5"}   # paquetes que se instalan con uv para poder aplicar la sobrescritura

# uv: instalador compatible con pip que sí permite sobrescribir dependencias (--override)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

LIBRERIAS = [   # (paquete para pip, nombre al importar)
    ("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"), ("scikit-learn", "sklearn"),
    ("pillow", "PIL"), ("requests", "requests"), ("soundfile", "soundfile"), ("rapidfuzz", "rapidfuzz"),
    ("plotly", "plotly"), ("nbformat", "nbformat"), ("ipywidgets", "ipywidgets"),
    ("torch", "torch"), ("torchvision", "torchvision"), ("transformers>=5.4", "transformers"),
    ("paddleocr>=3.5", "paddleocr"), ("faster-whisper>=1.2", "faster_whisper"),
]
reporte = []
for paquete, modulo in LIBRERIAS:
    if paquete in CON_SOBRESCRITURA:
        comando = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, "--only-binary", ":all:",
                   "-c", "restricciones.txt", "--override", "sobrescrituras.txt", paquete]
    else:
        comando = [sys.executable, "-m", "pip", "install", "-q", "-c", "restricciones.txt", paquete]
    r = subprocess.run(comando, capture_output=True, text=True)
    prueba = subprocess.run([sys.executable, "-c", f"import {modulo}; print(getattr({modulo}, '__version__', 'ok'))"],
                            capture_output=True, text=True)
    error = (r.stderr if r.returncode else prueba.stderr).strip().splitlines()
    reporte.append({"librería": paquete,
                    "instalación": "ok" if r.returncode == 0 else "ERROR",
                    "import": prueba.stdout.strip() if prueba.returncode == 0 else "ERROR",
                    "detalle": error[-1][:150] if (r.returncode or prueba.returncode) and error else ""})
    print(f"{paquete:22s} instalación: {reporte[-1]['instalación']:5s} import: {reporte[-1]['import']}")

fallas = [f for f in reporte if f["instalación"] != "ok" or f["import"] == "ERROR"]
print("\n" + ("Todo listo. Reinicia el kernel antes de continuar." if not fallas else
              "Fallaron estas librerías (copia este mensaje para revisarlo):"))
for f in fallas:
    print(f"  - {f['librería']}: {f['detalle']}")

Python del kernel: 3.14.7 → c:\Users\LENOVO\anaconda3\python.exe
numpy                  instalación: ok    import: 2.3.5
pandas                 instalación: ok    import: 3.0.3
scipy                  instalación: ok    import: 1.18.0
scikit-learn           instalación: ok    import: 1.9.0
pillow                 instalación: ok    import: 12.3.0
requests               instalación: ok    import: 2.34.2
soundfile              instalación: ok    import: 0.14.0
rapidfuzz              instalación: ok    import: 3.14.6
plotly                 instalación: ok    import: 6.7.0
nbformat               instalación: ok    import: 5.10.4
ipywidgets             instalación: ok    import: 8.1.7
torch                  instalación: ok    import: 2.14.1+cpu
torchvision            instalación: ok    import: 0.29.1+cpu
transformers>=5.4      instalación: ok    import: 5.20.1
paddleocr>=3.5         instalación: ok    import: 3.7.0
faster-whisper>=1.2    instalación: ok    import: 1.2.1

Todo listo. Reinicia 

## Importaciones, carpetas y versiones

Estructura de carpetas que crea el notebook:

```
datos/
├── propias/            fotos tomadas por mí (opcional)
├── raw/documentos/     imágenes originales + salida cruda del OCR (JSON, imagen anotada, recortes)
└── raw/audio/          clips WAV 16 kHz, audios originales y salidas CSV/SRT/TXT
```

`datos/raw/audio/`, `datos/raw/documentos/ocr/` y `datos/propias/` van en `.gitignore`: son pesados o pueden contener datos sin seudonimizar.

In [6]:
import hashlib
import io
import logging
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from collections import Counter
from datetime import date
from html import escape
from pathlib import Path
from urllib.parse import quote

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"   # antes de importar PaddleOCR: no probar servidores
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
for registro in ("paddlex", "paddleocr"):
    logging.getLogger(registro).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import plotly.express as px
import requests
import soundfile as sf
import torch
from IPython.display import HTML, display
from PIL import Image
from scipy.signal import resample_poly

pd.set_option("display.max_colwidth", 90)

if sys.version_info[:2] != (3, 14):
    warnings.warn(f"Notebook preparado para Python 3.14.7; este kernel es {sys.version.split()[0]}")

PROYECTO = Path.cwd()
DATOS = PROYECTO / "datos"
DIR_DOC = DATOS / "raw" / "documentos"
DIR_AUD = DATOS / "raw" / "audio"
for carpeta in [DIR_DOC / "ocr", DIR_AUD / "originales", DIR_AUD / "salidas" / "crudo", DATOS / "propias"]:
    carpeta.mkdir(parents=True, exist_ok=True)

import faster_whisper
import paddleocr
import transformers
from scipy.fft import dct
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

HAY_GPU = torch.cuda.is_available()
VERSIONES = {
    "fecha_extraccion": date.today().isoformat(),
    "python": sys.version.split()[0],
    "paddleocr": paddleocr.__version__,
    "transformers": transformers.__version__,
    "faster-whisper": faster_whisper.__version__,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if HAY_GPU else "no (CPU)",
}


def ruta_relativa(ruta):
    return Path(ruta).resolve().relative_to(PROYECTO.resolve()).as_posix()


def huella(ruta):
    "Hash SHA-256 del archivo: dos archivos idénticos tienen la misma huella."
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


pd.Series(VERSIONES).to_frame("valor")

,valor
fecha_extraccion,2026-10-25
python,3.14.7
paddleocr,3.7.0
transformers,5.20.1
faster-whisper,1.2.1
torch,2.14.1+cpu
gpu,no (CPU)


## Paso 2 · Documentos y OCR

### 2.1 · Acceso a Wikimedia Commons

Tres funciones: una para **ver** candidatos (galería), una para **descargar** un archivo y otra para leer su **licencia y autor** desde la API, de modo que la atribución se registra sin copiarla a mano.

In [9]:

API_COMMONS = "https://commons.wikimedia.org/w/api.php"
CABECERA = {"User-Agent": "proyecto-voces-documentos/1.0 (trabajo academico)"}


def consultar_api(params, intentos=5):
    "Consulta la API de Commons; si responde 429 o falla, espera lo que pide el servidor y reintenta."
    for intento in range(intentos):
        espera = 5 * 2 ** intento                       # 5, 10, 20, 40, 80 s
        try:
            r = requests.get(API_COMMONS, params=params | {"format": "json", "maxlag": 5},
                             headers=CABECERA, timeout=60)
            datos = r.json() if r.status_code == 200 else None
            if datos is not None and "error" not in datos:
                return datos
            motivo = f"HTTP {r.status_code}" if datos is None else datos["error"].get("code", "error")
            if r.headers.get("Retry-After", "").isdigit():
                espera = max(espera, int(r.headers["Retry-After"]))
        except (requests.RequestException, ValueError) as e:
            motivo = type(e).__name__
        print(f"  Commons no respondió ({motivo}); reintento en {espera} s…")
        time.sleep(espera)
    raise ConnectionError("Commons sigue limitando las consultas. Espera unos minutos y vuelve a ejecutar la celda.")


def ficha_commons(nombre):
    "Licencia, autor y URL de la ficha de un archivo de Commons."
    datos = consultar_api({"action": "query", "titles": f"File:{nombre}", "prop": "imageinfo",
                           "iiprop": "url|extmetadata"})
    pagina = next(iter(datos["query"]["pages"].values()))
    if "imageinfo" not in pagina:
        raise FileNotFoundError(f"No existe en Commons: {nombre}")
    info = pagina["imageinfo"][0]
    meta = info.get("extmetadata", {})
    texto = lambda clave: re.sub(r"<[^>]+>", "", meta.get(clave, {}).get("value", "")).strip()
    return {"licencia": texto("LicenseShortName") or "ver ficha", "autor": texto("Artist") or "ver ficha",
            "fuente_url": info["descriptionurl"]}


def bajar_de_commons(nombre, destino, ancho=None, intentos=5):
    "Descarga un archivo de Commons en tamaño estándar (máx. 1280 px), con pausas y reintentos. No repite descargas."
    if destino.exists():
        return destino
    params = {"action": "query", "titles": f"File:{nombre}", "prop": "imageinfo", "iiprop": "url"}
    if ancho:
        params["iiurlwidth"] = min(ancho, 1280)          # 1280 es un tamaño estándar de Commons
    pagina = next(iter(consultar_api(params)["query"]["pages"].values()))
    if "imageinfo" not in pagina:
        raise FileNotFoundError(f"No existe en Commons: {nombre}")
    info = pagina["imageinfo"][0]
    url = info.get("thumburl") or info["url"]
    for intento in range(intentos):
        espera = 30 * 2 ** intento                       # 30 s, 1 min, 2 min, 4 min, 8 min
        try:
            r = requests.get(url, headers=CABECERA, timeout=120)
            if r.status_code == 200:
                destino.write_bytes(r.content)
                time.sleep(5)                            # pausa entre descargas
                return destino
            motivo = f"HTTP {r.status_code}"
            if r.headers.get("Retry-After", "").isdigit():
                espera = max(espera, int(r.headers["Retry-After"]))
        except requests.RequestException as e:
            motivo = type(e).__name__
        print(f"  {nombre}: falló la descarga ({motivo}); reintento en {espera} s…")
        time.sleep(espera)
    raise ConnectionError(f"No se pudo descargar {nombre}")


def galeria(consulta, n=12, tipo="bitmap"):
    "Muestra miniaturas de Commons con su nombre exacto y licencia. No descarga nada."
    try:
        datos = consultar_api({"action": "query", "generator": "search", "gsrnamespace": 6, "gsrlimit": n,
                               "gsrsearch": f"{consulta} filetype:{tipo}", "prop": "imageinfo",
                               "iiprop": "url|extmetadata", "iiurlwidth": 220})
    except ConnectionError as e:
        display(HTML(f"<h4>{escape(consulta)}</h4><i>{escape(str(e))}</i>"))
        return
    paginas = sorted(datos.get("query", {}).get("pages", {}).values(), key=lambda p: p.get("index", 0))
    tarjetas = []
    for p in paginas:
        info = p["imageinfo"][0]
        licencia = re.sub(r"<[^>]+>", "", info.get("extmetadata", {}).get("LicenseShortName", {}).get("value", "?"))
        miniatura = info.get("thumburl", "")
        tarjetas.append(
            '<div style="width:230px;display:inline-block;vertical-align:top;margin:6px;font-size:11px">'
            + (f'<a href="{info["descriptionurl"]}" target="_blank"><img src="{miniatura}" '
               'style="max-width:220px;max-height:220px"></a><br>' if miniatura else "")
            + f'<code style="user-select:all">{escape(p["title"].removeprefix("File:"))}</code>'
            + f'<br><i>{escape(licencia)}</i></div>')
    display(HTML(f"<h4>{escape(consulta)}</h4>" + ("".join(tarjetas) or "<i>sin resultados</i>")))


print("Funciones listas:", [f for f in ("consultar_api", "ficha_commons", "bajar_de_commons", "galeria") if f in globals()])

Funciones listas: ['consultar_api', 'ficha_commons', 'bajar_de_commons', 'galeria']


### 2.2 · Elegir los documentos

La celda muestra miniaturas bajo cada búsqueda. Debajo de cada una aparece el **nombre exacto** (un clic lo selecciona completo para copiarlo) y la **licencia**. Al hacer clic en la imagen se abre su ficha en el navegador para verla en grande.

Criterios para aceptar un candidato:

1. Tiene texto legible a simple vista.
2. Es peruano.
3. Corresponde al tipo: **impreso** (letra de imprenta, nítido), **foto** (cartel o letrero fotografiado con sombra o inclinación) o **manuscrito** (escrito a mano).
4. No muestra datos personales de particulares (DNI, domicilios, firmas de personas privadas).

Elijo unos 10 por tipo (31 en total) para tener reserva si alguno se descarta en la limpieza. Para cambiar las búsquedas, edita `BUSQUEDAS`.

In [10]:
BUSQUEDAS = {
    "impreso":    ["Peru newspaper", "El Comercio Lima", "Peru poster", "Peru banknote", "Peru stamp", "decreto Perú"],
    "manuscrito": ["Guaman Poma", "Peru manuscript", "Peru letter handwritten", "colonial manuscript Lima", "carta Perú siglo XIX"],
}
for tipo, consultas in BUSQUEDAS.items():
    display(HTML(f"<h3>{tipo.upper()}</h3>"))
    for consulta in consultas:
        galeria(consulta)
        time.sleep(3)          # pausa entre búsquedas para no saturar a Commons

### 2.3 · Catálogo y descarga

Cada imagen de Commons se descarga reescalada a un máximo de 1600 px de ancho.

In [33]:
CANDIDATOS = [
    # ── FOTOS (10) ──
    ("foto", "Letrero de transporte por Lima.jpg"),
    ("foto", "Cartel informativo sobre el Templo del Sol.jpg"),
    ("foto", "Lima 2011 - letrero.jpg"),
    ("foto", "Letreros de calles en Lima 2.jpg"),
    ("foto", "Letreros de calles en Lima 3.jpg"),
    ("foto", "Letreros de calles en Lima 4.jpg"),
    ("foto", "Cartel Calle Espaderos.jpg"),
    ("foto", "Cartel Plazoleta Santa Catalina.jpg"),
    ("foto", "Cartel Calle Marqués.jpg"),
    ("foto", "Cartel Plazoleta Espinar.jpg"),

    # ── IMPRESOS (10) ──
    ("impreso", "El Comercio - Lima. Perú y Ecuador. 3-3-1842.jpg"),
    ("impreso", "El Comercio Issue 12.jpg"),
    ("impreso", "Cedula Real 1551 Universidad de San Marcos.png"),
    ("impreso", "El Comercio, 24 de mayo de 1940.jpg"),
    ("impreso", "Billete de 1870 de 5 soles del Banco Nacional del Perú.jpg"),
    ("impreso", "100 peruvian pesos.jpg"),
    ("impreso", "1000000 Intis 1990(Front).jpg"),
    ("impreso", "Stamp of Peru - 1957 - Colnect 131362 - 2r Stamp Of 1857.jpeg"),
    ("impreso", "5000000 Intis 1990(Reverse).jpg"),
    ("impreso", "Banco Central de Reserva del Perú, mil soles de oro, 1969.jpg"),

    # ── MANUSCRITOS (9) ──
    ("manuscrito", "Manuscrito de Perú (F. Gil de Lemos).Texto.jpg"),
    ("manuscrito", "Manuscrito de Perú (F. Gil de Lemos).Índice.jpg"),
    ("manuscrito", "Relación de antigüedades deste reino del Pirú (1613).jpg"),
    ("manuscrito", "Capitulacón de Ayacucho2.jpg"),
    ("manuscrito", "Alma matinal.jpg"),
    ("manuscrito", "Real Cédula otorgando el escudo de armas de la ciudad de Los Reyes (Lima).jpg"),
    ("manuscrito", "Manuscrito de Perú (F. Gil de Lemos).Tabla.jpg"),
    ("manuscrito", "Huarochiri Capitulo 1 Coniraya Viracocha runacamac pachacamac.png"),
    ("manuscrito", "Capitulacion de potosi 1561.jpg"),
    ("manuscrito", "Francisco Gil de Taboada Lemos y Villamarín.JPG"),
]


assert CANDIDATOS, "CANDIDATOS está vacío: elige documentos en la galería y pega sus nombres aquí"

LICENCIA_PROPIA = "CC BY 4.0 (foto propia)"
registros = []
for n, (tipo, origen) in enumerate(CANDIDATOS, start=1):
    doc_id = f"doc_{n:03d}"
    try:
        if origen.startswith("propia:"):
            original = DATOS / "propias" / origen.removeprefix("propia:")
            imagen = DIR_DOC / f"{doc_id}{original.suffix.lower()}"
            if not imagen.exists():
                shutil.copy(original, imagen)
            ficha = {"licencia": LICENCIA_PROPIA, "autor": "autoría propia", "fuente_url": "",
                     "acceso": f"foto propia, original en datos/propias/{original.name}"}
            proveniencia = "Foto propia"
        else:
            imagen = bajar_de_commons(origen, DIR_DOC / f"{doc_id}{Path(origen).suffix.lower()}", ancho=1600)
            ficha = ficha_commons(origen) | {"acceso": "descarga pública de Wikimedia Commons (ancho máx. 1600 px)"}
            proveniencia = "Wikimedia Commons"
        registros.append({"id": doc_id, "tipo": tipo, "origen": origen, "archivo": imagen,
                          "fuente": f"{proveniencia} ({ficha['licencia']})", **ficha,
                          "hash": huella(imagen), "error": ""})
    except Exception as e:
        registros.append({"id": doc_id, "tipo": tipo, "origen": origen, "error": f"{type(e).__name__}: {e}"})

catalogo_docs = pd.DataFrame(registros)
catalogo_docs["error"] = catalogo_docs["error"].fillna("")
print(catalogo_docs.groupby("tipo").size().to_string())
catalogo_docs[["id", "tipo", "origen", "licencia", "error"]]

  Commons no respondió (HTTP 429); reintento en 5 s…
  Commons no respondió (ConnectTimeout); reintento en 5 s…
  Commons no respondió (ConnectTimeout); reintento en 5 s…
  Commons no respondió (HTTP 429); reintento en 10 s…
  Commons no respondió (ConnectTimeout); reintento en 5 s…
  Commons no respondió (HTTP 429); reintento en 13 s…
tipo
foto          10
impreso       10
manuscrito    10


,id,tipo,origen,licencia,error
0,doc_001,foto,Letrero de transporte por Lima.jpg,CC0,
1,doc_002,foto,Cartel informativo sobre el Templo del Sol.jpg,CC BY-SA 4.0,
2,doc_003,foto,Lima 2011 - letrero.jpg,CC BY-SA 2.0,
3,doc_004,foto,Letreros de calles en Lima 2.jpg,CC BY-SA 4.0,
4,doc_005,foto,Letreros de calles en Lima 3.jpg,CC BY-SA 4.0,
5,doc_006,foto,Letreros de calles en Lima 4.jpg,CC BY-SA 4.0,
6,doc_007,foto,Cartel Calle Espaderos.jpg,CC BY-SA 4.0,
7,doc_008,foto,Cartel Plazoleta Santa Catalina.jpg,CC BY-SA 4.0,
8,doc_009,foto,Cartel Calle Marqués.jpg,CC BY-SA 4.0,
9,doc_010,foto,Cartel Plazoleta Espinar.jpg,CC BY-SA 4.0,


### 2.4 · Configuración del OCR

| Parámetro | Valor | Motivo |
|---|---|---|
| Modelo | PP-OCRv6 medium: detector de líneas + reconocedor | Con `lang="es"` PaddleOCR 3.7 elige estos modelos, que cubren el alfabeto latino con tildes y ñ |
| Idioma | `lang="es"` | Español |
| Motor | `engine="transformers"` (PyTorch) | PaddlePaddle no tiene versión para Python 3.14; los modelos son los mismos |
| Dispositivo | GPU NVIDIA si existe; si no, CPU | Se detecta automáticamente |
| Preprocesamiento | reescalado a ≤ 1600 px de ancho | Tamaño suficiente para leer y rápido de procesar |
| Clasificar orientación de página / desdoblar hoja | desactivado | Las imágenes vienen derechas; desdoblar puede deformar las letras |
| Orientación por línea | activado solo en **fotos** | En las fotos el texto suele estar girado o inclinado |

Por cada documento guardo en `datos/raw/documentos/ocr/<id>/` el JSON crudo (textos, cajas y confianza), la imagen con las cajas dibujadas y un recorte por línea.

In [13]:
from paddleocr import PaddleOCR

DISPOSITIVO_OCR = "gpu:0" if HAY_GPU else "cpu"
CONFIG_OCR = {
    "impreso":    {"use_doc_orientation_classify": False, "use_doc_unwarping": False, "use_textline_orientation": False},
    "manuscrito": {"use_doc_orientation_classify": False, "use_doc_unwarping": False, "use_textline_orientation": False},
    "foto":       {"use_doc_orientation_classify": False, "use_doc_unwarping": False, "use_textline_orientation": True},
}
lector_general = PaddleOCR(lang="es", engine="transformers", device=DISPOSITIVO_OCR, **CONFIG_OCR["impreso"])
lector_fotos = PaddleOCR(lang="es", engine="transformers", device=DISPOSITIVO_OCR, **CONFIG_OCR["foto"])

DESCRIPCION_OCR = {
    tipo: (f"PaddleOCR {paddleocr.__version__} PP-OCRv6 medium (lang=es, engine=transformers, {DISPOSITIVO_OCR}, "
           f"≤1600 px, orientación de línea {'on' if cfg['use_textline_orientation'] else 'off'})")
    for tipo, cfg in CONFIG_OCR.items()
}
DESCRIPCION_OCR

<Response [404]>


2026-10-25 16:54:17,842 | INFO    | modelscope_hub.download | Downloading 6 files from PaddlePaddle/PP-OCRv6_medium_rec_safetensors@master


Downloading:   0%|          | 0/6 [00:00<?, ?file/s]

inference.yml:   0%|          | 0.00/151k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.76M [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

{'impreso': 'PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientación de línea off)',
 'manuscrito': 'PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientación de línea off)',
 'foto': 'PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientación de línea on)'}

In [34]:
def extraer_texto(doc):
    "OCR de un documento: guarda la salida cruda y los recortes, y devuelve una fila por línea leída."
    lector = lector_fotos if doc.tipo == "foto" else lector_general
    resultado = lector.predict(str(doc.archivo))[0]
    salida = DIR_DOC / "ocr" / doc.id
    salida.mkdir(parents=True, exist_ok=True)
    for guardar in (resultado.save_to_json, resultado.save_to_img):
        try:
            guardar(str(salida))
        except Exception as e:
            print(f"{doc.id}: no se guardó la salida cruda ({e})")
    cajas = np.asarray(resultado["rec_boxes"]).reshape(-1, 4).astype(int)
    pagina = Image.open(doc.archivo).convert("RGB")
    for k, (x0, y0, x1, y1) in enumerate(cajas):
        if x1 > x0 and y1 > y0:
            pagina.crop((x0, y0, x1, y1)).save(salida / f"linea_{k:03d}.png")
    return pd.DataFrame({"texto": resultado["rec_texts"], "confianza": np.round(resultado["rec_scores"], 3)})


lineas = {}
inicio = time.perf_counter()
for doc in catalogo_docs[catalogo_docs.error == ""].itertuples():
    lineas[doc.id] = extraer_texto(doc)
print(f"OCR de {len(lineas)} documentos en {time.perf_counter() - inicio:.0f} s")

catalogo_docs["n_lineas"] = catalogo_docs.id.map(lambda i: len(lineas.get(i, [])))
catalogo_docs["conf_mediana"] = catalogo_docs.id.map(
    lambda i: lineas[i].confianza.median() if len(lineas.get(i, [])) else np.nan)
catalogo_docs["texto_crudo"] = catalogo_docs.id.map(lambda i: "\n".join(lineas[i].texto) if i in lineas else "")
catalogo_docs["modelo_ocr"] = catalogo_docs.tipo.map(DESCRIPCION_OCR)
catalogo_docs[["id", "tipo", "n_lineas", "conf_mediana"]].round(2)

OCR de 30 documentos en 185 s


,id,tipo,n_lineas,conf_mediana
0,doc_001,foto,48,1.00
1,doc_002,foto,23,0.99
2,doc_003,foto,3,1.00
3,doc_004,foto,6,0.91
4,doc_005,foto,5,0.93
5,doc_006,foto,5,0.84
6,doc_007,foto,2,1.00
7,doc_008,foto,2,1.00
8,doc_009,foto,2,0.96
9,doc_010,foto,2,0.94


## Paso 3 · Audio y transcripción

### 3.0 · Herramientas de audio

Funciones propias para todo el flujo de audio:

| Función | Qué hace | Librería |
|---|---|---|
| `leer_audio` | Cualquier formato → señal mono a 16 kHz (lo que esperan los modelos de voz) | faster-whisper (PyAV), soundfile, scipy |
| `detectar_voz` | Tramos (inicio, fin) donde alguien habla | VAD Silero incluido en faster-whisper |
| `transcribir` | Señal → segmentos con tiempos y texto | faster-whisper |
| `huella_vocal` | Resume la voz de un tramo en 38 números (media y desviación de 19 coeficientes MFCC) | NumPy, SciPy |
| `separar_hablantes` | Agrupa las huellas y etiqueta H1, H2… en cada segmento | scikit-learn (clustering jerárquico de Ward y silueta) |
| `a_srt`, `a_txt` | Exportación a subtítulos y a texto legible por turnos | — |

In [15]:
SR = 16_000


def leer_audio(origen, sr_origen=None):
    "Ruta de archivo o arreglo numpy → float32 mono a 16 kHz."
    if isinstance(origen, (str, Path)):
        try:
            return faster_whisper.decode_audio(str(origen), sampling_rate=SR)
        except Exception:                         # respaldo para archivos que PyAV no abre
            origen, sr_origen = sf.read(str(origen), dtype="float32")
    x = np.asarray(origen, dtype=np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if sr_origen and sr_origen != SR:
        g = np.gcd(int(sr_origen), SR)
        x = resample_poly(x, SR // g, int(sr_origen) // g).astype(np.float32)
    return x


def detectar_voz(x, pausa_min=0.5):
    "Tramos de voz en segundos según el VAD Silero."
    from faster_whisper.vad import VadOptions, get_speech_timestamps
    tramos = get_speech_timestamps(x, VadOptions(min_silence_duration_ms=int(pausa_min * 1000)))
    return [(t["start"] / SR, t["end"] / SR) for t in tramos]


modelo_asr = faster_whisper.WhisperModel("large-v3-turbo", device="cuda" if HAY_GPU else "cpu",
                                         compute_type="float16" if HAY_GPU else "int8")
CONFIG_ASR = {"language": "es", "beam_size": 5, "vad_filter": True,
              "vad_parameters": {"min_silence_duration_ms": 300}}


def transcribir(x):
    "Segmentos [{inicio, fin, texto, hablante}] de un audio de cualquier largo."
    segs, _ = modelo_asr.transcribe(x, **CONFIG_ASR)
    return [{"inicio": round(s.start, 2), "fin": round(s.end, 2), "texto": s.text.strip(), "hablante": None}
            for s in segs if s.text.strip()]


def _banco_mel(n_filtros=40, n_fft=512, fmin=60, fmax=7600):
    "Filtros triangulares en escala mel: imitan cómo el oído agrupa las frecuencias."
    hz_a_mel = lambda f: 2595 * np.log10(1 + f / 700)
    mel_a_hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    puntos = mel_a_hz(np.linspace(hz_a_mel(fmin), hz_a_mel(fmax), n_filtros + 2))
    bins = np.floor((n_fft + 1) * puntos / SR).astype(int)
    banco = np.zeros((n_filtros, n_fft // 2 + 1))
    for i in range(n_filtros):
        a, b, c = bins[i], bins[i + 1], bins[i + 2]
        if b > a:
            banco[i, a:b] = (np.arange(a, b) - a) / (b - a)
        if c > b:
            banco[i, b:c] = (c - np.arange(b, c)) / (c - b)
    return banco


BANCO_MEL = _banco_mel()


def huella_vocal(tramo):
    """Huella de una voz: media y desviación de 19 coeficientes MFCC calculados cada 10 ms.
    Los MFCC describen el timbre (forma del tracto vocal), que cambia de una persona a otra."""
    x = np.append(tramo[:1], tramo[1:] - 0.97 * tramo[:-1]).astype(np.float64)   # pre-énfasis
    largo, salto = 400, 160                                                       # ventanas de 25 ms cada 10 ms
    x = np.pad(x, (0, max(0, largo - len(x))))
    n = 1 + (len(x) - largo) // salto
    ventanas = x[np.arange(largo)[None, :] + salto * np.arange(n)[:, None]] * np.hamming(largo)
    potencia = np.abs(np.fft.rfft(ventanas, 512)) ** 2 / 512
    energia = potencia.sum(axis=1)
    potencia = potencia[energia >= np.percentile(energia, 30)]                   # quita las ventanas más silenciosas
    mfcc = dct(np.log(potencia @ BANCO_MEL.T + 1e-10), type=2, norm="ortho", axis=1)[:, 1:20]
    return np.concatenate([mfcc.mean(axis=0), mfcc.std(axis=0)])


def separar_hablantes(x, segs, n_hablantes=None, dur_min=1.0, k_max=4, silueta_min=0.5):
    """Etiqueta cada segmento con H1, H2…
    - Con `n_hablantes` conocido, forma exactamente ese número de grupos.
    - Si es None, prueba de 2 a `k_max` grupos y se queda con el de mejor coeficiente de silueta;
      si ninguno supera `silueta_min`, concluye que hay un solo hablante.
    Los segmentos de menos de `dur_min` s no participan en el agrupamiento (su huella es poco
    fiable) y después se asignan al grupo más cercano."""
    if len(segs) < 2 or n_hablantes == 1:
        return [s | {"hablante": "H1"} for s in segs]
    huellas = np.vstack([huella_vocal(x[int(s["inicio"] * SR):int(s["fin"] * SR)]) for s in segs])
    huellas = StandardScaler().fit_transform(huellas)
    largos = np.array([s["fin"] - s["inicio"] >= dur_min for s in segs])
    if largos.sum() < 3:
        largos[:] = True
    base = huellas[largos]
    if n_hablantes:
        k = min(n_hablantes, len(base))
    else:
        k, mejor = 1, silueta_min
        for candidato in range(2, min(k_max, len(base) - 1) + 1):
            etiquetas = AgglomerativeClustering(n_clusters=candidato, linkage="ward").fit_predict(base)
            silueta = silhouette_score(base, etiquetas)
            if silueta > mejor:
                k, mejor = candidato, silueta
    if k == 1:
        return [s | {"hablante": "H1"} for s in segs]
    etiquetas = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(base)
    centros = np.vstack([base[etiquetas == g].mean(axis=0) for g in range(k)])
    grupos = ((huellas[:, None, :] - centros[None]) ** 2).sum(axis=2).argmin(axis=1)
    grupos[largos] = etiquetas
    orden = {g: f"H{i}" for i, g in enumerate(dict.fromkeys(grupos), start=1)}   # H1 = quien habla primero
    return [s | {"hablante": orden[g]} for s, g in zip(segs, grupos)]


def _tiempo(t, sep=","):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"


def a_srt(segs):
    return "\n".join(f"{k}\n{_tiempo(s['inicio'])} --> {_tiempo(s['fin'])}\n"
                     f"{'[' + s['hablante'] + '] ' if s['hablante'] else ''}{s['texto']}\n"
                     for k, s in enumerate(segs, start=1))


def turnos(segs):
    "Agrupa segmentos consecutivos del mismo hablante: [(inicio, hablante, texto)]."
    salida = []
    for s in segs:
        if salida and salida[-1][1] == s["hablante"]:
            salida[-1][2].append(s["texto"])
        else:
            salida.append((s["inicio"], s["hablante"], [s["texto"]]))
    return [(t, h, " ".join(textos)) for t, h, textos in salida]


def a_txt(segs):
    return "\n\n".join(f"[{_tiempo(t, '.')[:8]}] {h + ': ' if h else ''}{texto}" for t, h, texto in turnos(segs))

### 3.1 · Entrevistas de Wikimedia Commons (opcional, recomendado)

La galería muestra audios de Commons. Escucha los candidatos en su ficha (clic en «ficha») y agrega a `ENTREVISTAS` los que sean **entrevistas en español peruano** con licencia libre, indicando **cuántas personas hablan**: al escucharla lo sabes, y ese dato hace más fiable la separación de hablantes. Si no estás seguro, pon `None` y se estima automáticamente.

In [16]:
def galeria_audio(consulta, n=10):
    "Lista de audios de Commons con duración, licencia y enlace a su ficha."
    r = requests.get(API_COMMONS, params={"action": "query", "generator": "search", "gsrnamespace": 6, "gsrlimit": n,
                                          "gsrsearch": f"{consulta} filetype:audio", "prop": "imageinfo",
                                          "iiprop": "url|extmetadata|size", "format": "json"},
                     headers=CABECERA, timeout=60)
    r.raise_for_status()
    filas = []
    for p in sorted(r.json().get("query", {}).get("pages", {}).values(), key=lambda p: p.get("index", 0)):
        info = p["imageinfo"][0]
        licencia = re.sub(r"<[^>]+>", "", info.get("extmetadata", {}).get("LicenseShortName", {}).get("value", "?"))
        filas.append(f'<li><a href="{info["descriptionurl"]}" target="_blank">ficha</a> · '
                     f'<code style="user-select:all">{escape(p["title"].removeprefix("File:"))}</code> · '
                     f'{info.get("duration", 0) / 60:.1f} min · <i>{escape(licencia)}</i></li>')
    display(HTML(f"<h4>{escape(consulta)}</h4><ul>{''.join(filas) or '<i>sin resultados</i>'}</ul>"))


for consulta in ["entrevista Perú", "entrevista Lima", "testimonio Perú", "historia oral Perú"]:
    galeria_audio(consulta)

In [17]:
ENTREVISTAS = [   # (nombre exacto en Commons, descripción sin nombres de personas, n.º de hablantes o None)
    ("Entrevista 1,.wav", "entrevista en Perú (1)", None),
    ("Entrevista 2.wav", "entrevista en Perú (2)", None),
    ("Entrevista 3.wav", "entrevista en Perú (3)", None),
    ("Entrevista 4.wav", "entrevista en Perú (4)", None),
    ("Audio de la entrevista a una chica que participaba de la marcha.ogg", "entrevista en una marcha", None),
]
audios = []
for k, (nombre, descripcion, n_hablantes) in enumerate(ENTREVISTAS, start=1):
    destino = DIR_AUD / "originales" / f"entrevista_{k:02d}{Path(nombre).suffix.lower()}"
    try:
        bajar_de_commons(nombre, destino)
        audios.append({"original": destino, "fuente": "Wikimedia Commons", **ficha_commons(nombre),
                       "descripcion": descripcion, "acceso": "descarga pública de Wikimedia Commons",
                       "separar": True, "n_conocido": np.nan if n_hablantes is None else n_hablantes,
                       "referencia_fuente": ""})
        print(f"✓ {descripcion}")
    except Exception as e:
        print(f"✗ {nombre}: {type(e).__name__}: {e}")
print(f"\n{len(audios)} entrevistas de Commons listas")

✓ entrevista en Perú (1)
✓ entrevista en Perú (2)
✓ entrevista en Perú (3)
✓ entrevista en Perú (4)
✓ entrevista en una marcha

5 entrevistas de Commons listas


### 3.2 · OpenSLR 73: lecturas y diálogos

OpenSLR 73 (*Crowdsourced high-quality Peruvian Spanish speech*, Google, CC BY-SA 4.0) trae miles de frases leídas por hablantes peruanos, cada una con su transcripción oficial. Cada zip pesa unos 900 MB, así que no lo descargo completo: el objeto `ZipPorHTTP` le pide al servidor solo los bytes de las frases que necesito (peticiones HTTP *Range*).

Con esas frases armo dos tipos de clip, siempre con voz real:

- **Diálogos** (2 clips): frases de una mujer y un hombre intercaladas, con 0,7 s de pausa. Como sé quién dice cada frase, sirven para verificar la separación de hablantes. Cubren el requisito de audio multihablante si no agregas entrevistas.
- **Lecturas**: varias frases de un mismo hablante unidas en un clip de ~1 minuto, hasta completar 30 minutos y 20 clips.

Las transcripciones oficiales se guardan para la evaluación de la Parte 2.

In [18]:
class ZipPorHTTP(io.RawIOBase):
    "Archivo remoto de solo lectura: descarga bloques de 1 MB a demanda con peticiones HTTP Range."
    BLOQUE = 1 << 20

    def __init__(self, url):
        r = requests.head(url, headers=CABECERA, allow_redirects=True, timeout=60)
        r.raise_for_status()
        self.url, self.tamano, self.pos, self.cache = r.url, int(r.headers["Content-Length"]), 0, {}

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, desplazamiento, desde=io.SEEK_SET):
        self.pos = {io.SEEK_SET: 0, io.SEEK_CUR: self.pos, io.SEEK_END: self.tamano}[desde] + desplazamiento
        return self.pos

    def _bloque(self, k):
        if k not in self.cache:
            if len(self.cache) > 64:
                self.cache.clear()
            ini = k * self.BLOQUE
            fin = min(ini + self.BLOQUE, self.tamano) - 1
            r = requests.get(self.url, headers=CABECERA | {"Range": f"bytes={ini}-{fin}"}, timeout=120)
            if r.status_code != 206:
                raise IOError(f"El servidor no aceptó la descarga parcial (código {r.status_code})")
            self.cache[k] = r.content
        return self.cache[k]

    def read(self, n=-1):
        n = self.tamano - self.pos if n is None or n < 0 else min(n, self.tamano - self.pos)
        partes = []
        while n > 0:
            k, desfase = divmod(self.pos, self.BLOQUE)
            trozo = self._bloque(k)[desfase:desfase + n]
            if not trozo:
                break
            partes.append(trozo)
            self.pos += len(trozo)
            n -= len(trozo)
        return b"".join(partes)


URL_OPENSLR73 = "https://www.openslr.org/resources/73/es_pe_{}.zip"
corpus73 = {}
for sexo in ["female", "male"]:
    z = zipfile.ZipFile(ZipPorHTTP(URL_OPENSLR73.format(sexo)))
    frases = pd.DataFrame([l.split("\t")[:2] for l in z.read("line_index.tsv").decode("utf-8").strip().splitlines()],
                          columns=["utt", "texto"])
    frases["texto"] = frases.texto.str.strip()
    frases["hablante"] = frases.utt.str.rsplit("_", n=1).str[0]
    frases["posicion"] = [z.getinfo(f"{u}.wav").header_offset for u in frases.utt]   # leer en orden = menos peticiones
    corpus73[sexo] = (z, frases.sort_values("posicion"))
    print(f"{sexo}: {len(frases)} frases de {frases.hablante.nunique()} hablantes")


def frase_73(sexo, utt):
    z, _ = corpus73[sexo]
    x, sr = sf.read(io.BytesIO(z.read(f"{utt}.wav")), dtype="float32")
    return leer_audio(x, sr)

female: 2529 frases de 18 hablantes
male: 2918 frases de 20 hablantes


### 3.3 · Transcripción

Cada clip pasa por tres etapas:

1. **Detección de voz (VAD Silero)**: mide cuánto del clip es voz. Sin esto, Whisper tiende a inventar frases en los silencios.
2. **Transcripción** con faster-whisper `large-v3-turbo` (int8 en CPU, float16 en GPU), `beam_size=5`, VAD activado e idioma fijo en español.
3. **Separación de hablantes** en entrevistas y diálogos: huella MFCC de cada segmento y agrupamiento jerárquico. En los diálogos fijo 2 hablantes; en las entrevistas uso el número indicado en 3.1 o, si es `None`, el que da mejor coeficiente de silueta. Las lecturas son de un solo hablante.

Las salidas sin seudonimizar quedan solo en `datos/raw/audio/salidas/crudo/`.

In [21]:
audios = [a for a in audios if a["fuente"] == "Wikimedia Commons"]   # si se reejecuta, no duplica diálogos ni lecturas


def frase_segura(sexo, utt, intentos=6):
    "Lee una frase de OpenSLR; si la conexión se corta, espera y reintenta."
    for intento in range(intentos):
        try:
            return frase_73(sexo, utt)
        except Exception as e:
            espera = 10 * 2 ** intento                   # 10 s, 20 s, 40 s, 80 s, 160 s, 320 s
            print(f"  {utt}: {type(e).__name__}; reintento en {espera} s…")
            time.sleep(espera)
    raise ConnectionError("OpenSLR no respondió tras varios intentos. Espera unos minutos y vuelve a ejecutar esta celda.")


azar = random.Random(2026)
hablantes = {sexo: azar.sample(sorted(f.hablante.unique()), f.hablante.nunique()) for sexo, (_, f) in corpus73.items()}
DIR_ORIG = DIR_AUD / "originales"

# Diálogos: una mujer y un hombre alternando frases (si ya se armaron antes, se reutilizan)
for k in range(2):
    mujer, hombre = hablantes["female"].pop(), hablantes["male"].pop()
    destino = DIR_ORIG / f"dialogo_{k + 1}.wav"
    archivo_guion = DIR_ORIG / f"dialogo_{k + 1}_guion.csv"
    if destino.exists() and archivo_guion.exists():
        guion = pd.read_csv(archivo_guion).to_dict("records")
        estado = "reutilizado"
    else:
        fm = corpus73["female"][1].query("hablante == @mujer").head(6)
        fh = corpus73["male"][1].query("hablante == @hombre").head(6)
        pedazos, guion, t = [], [], 0.0
        for a, b in zip(fm.itertuples(), fh.itertuples()):
            for sexo, f, rol in [("female", a, "Hablante A"), ("male", b, "Hablante B")]:
                x = frase_segura(sexo, f.utt)
                guion.append({"inicio": round(t, 2), "fin": round(t + len(x) / SR, 2), "hablante": rol, "texto": f.texto})
                pedazos += [x, np.zeros(int(0.7 * SR), np.float32)]
                t += len(x) / SR + 0.7
        sf.write(destino, np.concatenate(pedazos), SR)
        pd.DataFrame(guion).to_csv(archivo_guion, index=False)
        estado = "listo"
    audios.append({"original": destino, "fuente": "OpenSLR 73 (diálogo armado)", "licencia": "CC BY-SA 4.0",
                   "fuente_url": "https://www.openslr.org/73/", "autor": "Google",
                   "descripcion": f"Diálogo armado: frases intercaladas de {mujer} y {hombre}",
                   "acceso": "es_pe_female.zip y es_pe_male.zip de openslr.org/73, lectura parcial (HTTP Range)",
                   "separar": True, "n_conocido": 2, "referencia_fuente": " ".join(str(g["texto"]) for g in guion)})
    print(f"✓ diálogo {k + 1} {estado} ({guion[-1]['fin'] / 60:.1f} min)")

# Lecturas: un hablante por clip hasta completar la meta (las ya armadas se reutilizan)
META_SEGUNDOS, META_CLIPS = 30 * 60 * 1.05, 22          # 5 % y 2 clips de margen por si hay descartes
acumulado = sum(len(leer_audio(a["original"])) / SR for a in audios)
largo_clip = float(np.clip((META_SEGUNDOS - acumulado) / max(META_CLIPS - len(audios), 1), 30, 90))
print(f"Entrevistas + diálogos: {acumulado / 60:.1f} min · lecturas de ~{largo_clip:.0f} s hasta completar la meta")
turno = 0
while acumulado < META_SEGUNDOS or len(audios) < META_CLIPS:
    sexo = ["female", "male"][turno % 2]
    turno += 1
    if not hablantes[sexo]:
        if not any(hablantes.values()):
            break
        continue
    h = hablantes[sexo].pop()
    destino = DIR_ORIG / f"lectura_{h}.wav"
    archivo_textos = DIR_ORIG / f"lectura_{h}.txt"
    if destino.exists() and archivo_textos.exists():
        senal = leer_audio(destino)
        textos = archivo_textos.read_text(encoding="utf-8").splitlines()
        estado = "reutilizada"
    else:
        pedazos, textos = [], []
        for f in corpus73[sexo][1].query("hablante == @h").itertuples():
            pedazos += [frase_segura(sexo, f.utt), np.zeros(int(0.6 * SR), np.float32)]
            textos.append(f.texto)
            if sum(map(len, pedazos)) / SR >= largo_clip:
                break
        senal = np.concatenate(pedazos)
        sf.write(destino, senal, SR)
        archivo_textos.write_text("\n".join(textos), encoding="utf-8")
        estado = "lista"
    acumulado += len(senal) / SR
    audios.append({"original": destino, "fuente": "OpenSLR 73", "licencia": "CC BY-SA 4.0",
                   "fuente_url": "https://www.openslr.org/73/", "autor": "Google",
                   "descripcion": f"Lectura: {len(textos)} frases de {'una mujer' if sexo == 'female' else 'un hombre'} ({h})",
                   "acceso": f"es_pe_{sexo}.zip de openslr.org/73, lectura parcial (HTTP Range)",
                   "separar": False, "n_conocido": 1, "referencia_fuente": " ".join(textos)})
    print(f"✓ lectura {len(audios)} {estado}: {acumulado / 60:.1f} de {META_SEGUNDOS / 60:.1f} min")

catalogo_aud = pd.DataFrame(audios)
catalogo_aud["hash"] = catalogo_aud.original.map(huella)
catalogo_aud["id"] = [f"clip_{n:03d}" for n in range(1, len(catalogo_aud) + 1)]
catalogo_aud["archivo_audio"] = [DIR_AUD / f"{i}.wav" for i in catalogo_aud.id]
for a in catalogo_aud.itertuples():
    sf.write(a.archivo_audio, leer_audio(a.original), SR)          # todos como WAV mono 16 kHz
catalogo_aud["duracion_seg"] = catalogo_aud.archivo_audio.map(lambda p: round(sf.info(p).duration, 1))
print(f"\n{len(catalogo_aud)} clips · {catalogo_aud.duracion_seg.sum() / 60:.1f} min")
catalogo_aud[["id", "fuente", "descripcion", "duracion_seg"]]

✓ diálogo 1 reutilizado (1.4 min)
✓ diálogo 2 reutilizado (1.5 min)
Entrevistas + diálogos: 12.7 min · lecturas de ~75 s hasta completar la meta
✓ lectura 8 lista: 14.0 de 31.5 min
✓ lectura 9 lista: 15.2 de 31.5 min
✓ lectura 10 lista: 16.5 de 31.5 min
✓ lectura 11 lista: 17.8 de 31.5 min
✓ lectura 12 lista: 19.1 de 31.5 min
✓ lectura 13 lista: 20.4 de 31.5 min
✓ lectura 14 lista: 21.8 de 31.5 min
✓ lectura 15 lista: 23.1 de 31.5 min
✓ lectura 16 lista: 24.4 de 31.5 min
✓ lectura 17 lista: 25.7 de 31.5 min
✓ lectura 18 lista: 27.0 de 31.5 min
✓ lectura 19 lista: 28.3 de 31.5 min
✓ lectura 20 lista: 29.6 de 31.5 min
  pem_09697_00049835770: ConnectTimeout; reintento en 10 s…
✓ lectura 21 lista: 30.9 de 31.5 min
✓ lectura 22 lista: 32.2 de 31.5 min

22 clips · 32.2 min


,id,fuente,descripcion,duracion_seg
0,clip_001,Wikimedia Commons,entrevista en Perú (1),196.0
1,clip_002,Wikimedia Commons,entrevista en Perú (2),96.1
2,clip_003,Wikimedia Commons,entrevista en Perú (3),77.4
3,clip_004,Wikimedia Commons,entrevista en Perú (4),163.9
4,clip_005,Wikimedia Commons,entrevista en una marcha,54.3
5,clip_006,OpenSLR 73 (diálogo armado),Diálogo armado: frases intercaladas de pef_02484 y pem_06136,83.2
6,clip_007,OpenSLR 73 (diálogo armado),Diálogo armado: frases intercaladas de pef_01523 y pem_00610,88.2
7,clip_008,OpenSLR 73,Lectura: 14 frases de una mujer (pef_00610),78.3
8,clip_009,OpenSLR 73,Lectura: 11 frases de un hombre (pem_03034),76.8
9,clip_010,OpenSLR 73,Lectura: 12 frases de una mujer (pef_01208),76.4


In [24]:
DESCRIPCION_ASR = f"faster-whisper {faster_whisper.__version__} large-v3-turbo ({'float16' if HAY_GPU else 'int8'}, beam 5, VAD on, idioma=es)"
CON_HABLANTES = " + separación de hablantes (MFCC + clustering Ward)"

segmentos_crudos, segundos_voz = {}, {}
inicio = time.perf_counter()
for n, clip in enumerate(catalogo_aud.itertuples(), start=1):
    x = leer_audio(clip.archivo_audio)
    segundos_voz[clip.id] = sum(b - a for a, b in detectar_voz(x))
    segs = transcribir(x) if segundos_voz[clip.id] >= 1.0 else []
    if clip.separar and segs:
        segs = separar_hablantes(x, segs, n_hablantes=None if pd.isna(clip.n_conocido) else int(clip.n_conocido))
    segmentos_crudos[clip.id] = segs
    (DIR_AUD / "salidas" / "crudo" / f"{clip.id}.srt").write_text(a_srt(segs), encoding="utf-8")
    print(f"✓ {clip.id} ({n}/{len(catalogo_aud)}): {len(segs)} segmentos · "
          f"{(time.perf_counter() - inicio) / 60:.1f} min transcurridos")

minutos_proceso = (time.perf_counter() - inicio) / 60
print(f"\n{catalogo_aud.duracion_seg.sum() / 60:.1f} min de audio procesados en {minutos_proceso:.1f} min")
catalogo_aud["modelo_asr"] = np.where(catalogo_aud.separar, DESCRIPCION_ASR + CON_HABLANTES, DESCRIPCION_ASR)
catalogo_aud["segundos_voz"] = catalogo_aud.id.map(segundos_voz).round(1)

for clip in catalogo_aud[catalogo_aud.fuente.str.contains("diálogo")].itertuples():
    detectados = len({s["hablante"] for s in segmentos_crudos[clip.id]})
    print(f"{clip.id}: {len(set(s['hablante'] for s in segmentos_crudos[clip.id]))} hablantes detectados (reales: 2)")
print(a_txt(segmentos_crudos[catalogo_aud.id[catalogo_aud.fuente.str.contains('diálogo')].iloc[0]])[:600])

✓ clip_001 (1/22): 36 segmentos · 1.2 min transcurridos
✓ clip_002 (2/22): 8 segmentos · 1.6 min transcurridos
✓ clip_003 (3/22): 12 segmentos · 1.9 min transcurridos
✓ clip_004 (4/22): 37 segmentos · 2.7 min transcurridos
✓ clip_005 (5/22): 11 segmentos · 2.9 min transcurridos
✓ clip_006 (6/22): 15 segmentos · 3.4 min transcurridos
✓ clip_007 (7/22): 12 segmentos · 3.6 min transcurridos
✓ clip_008 (8/22): 14 segmentos · 3.9 min transcurridos
✓ clip_009 (9/22): 11 segmentos · 4.1 min transcurridos
✓ clip_010 (10/22): 12 segmentos · 4.3 min transcurridos
✓ clip_011 (11/22): 13 segmentos · 4.5 min transcurridos
✓ clip_012 (12/22): 12 segmentos · 4.7 min transcurridos
✓ clip_013 (13/22): 12 segmentos · 4.9 min transcurridos
✓ clip_014 (14/22): 11 segmentos · 5.1 min transcurridos
✓ clip_015 (15/22): 10 segmentos · 5.3 min transcurridos
✓ clip_016 (16/22): 8 segmentos · 5.5 min transcurridos
✓ clip_017 (17/22): 13 segmentos · 5.7 min transcurridos
✓ clip_018 (18/22): 12 segmentos · 5.9 min

## Paso 4 · Seudonimización

Ningún nombre, DNI ni dato personal real puede publicarse. El procedimiento:

1. Buscar en todos los textos secuencias con forma de nombre propio (dos o más palabras con mayúscula inicial) y listarlas por frecuencia.
2. Revisar la lista a mano y pasar a `REEMPLAZOS` los nombres de **personas** (los de lugares e instituciones se quedan). El reemplazo compara por similitud (rapidfuzz, ≥ 85 %), así que también captura versiones mal escritas por el OCR o el ASR, como «Vázquez» por «Vásquez».
3. Reemplazar con expresiones regulares DNI (8 dígitos), celulares (9 dígitos empezando por 9) y correos.
4. Contar los reemplazos por tipo y por corpus.

Decisión: también se seudonimizan personas históricas que aparezcan en los documentos, porque la regla no distingue. La autoría que exigen las licencias se mantiene en `fuentes.csv` y `atribucion.csv`, que no son textos del corpus.

In [36]:
PATRON_NOMBRE = r"\b[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+(?:de\s+(?:la\s+)?|del\s+)?[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)+"
todos_los_textos = (list(catalogo_docs.texto_crudo.fillna(""))
                    + [" ".join(s["texto"] for s in segs) for segs in segmentos_crudos.values()])
posibles_nombres = pd.Series([m for t in todos_los_textos for m in re.findall(PATRON_NOMBRE, t)], dtype=str)
posibles_nombres.value_counts().head(60).rename("apariciones").to_frame()

,apariciones
Estados Unidos,5
Pierre Venegas,4
Goleta Nacional Fortuna,3
Juan Rulfo,3
Templo del Sol,2
Goleta Ecuatoriana Guayaquileña,2
Universidad de San Marcos,2
El Perú,2
San Marcos,2
El Llano,2


In [38]:
from rapidfuzz import fuzz

REEMPLAZOS = {          # nombre real → seudónimo
    "Pierre Venegas": "Persona_01",
    "Juan Rulfo": "Persona_02",
    "Rufino Cuervo": "Persona_03",
    "Cuervo": "Persona_03",
    "Jeneral Fiores": "Persona_04",
    "Señor Dorado": "Persona_05",
    "Santa Cruz": "Persona_06",
    "Simon Simener de Villalba": "Persona_07",
    "dTorefReroxrill": "Persona_08",
    "Fifranc": "Persona_09",
}

PATRONES = {
    "[CORREO]": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "[TELEFONO]": r"\b9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b",
    "[DNI]": r"\b\d{8}\b",
}
reemplazos_hechos = {"documentos": Counter(), "audio": Counter()}


def reemplazar_nombre(texto, nombre, seudonimo, similitud=85):
    "Recorre el texto con una ventana de tantas palabras como el nombre y reemplaza las que se le parecen."
    palabras = list(re.finditer(r"\w+", texto))
    largo = len(nombre.split())
    objetivo = nombre.lower().replace(" ", "")
    tramos, i = [], 0
    while i <= len(palabras) - largo:
        ventana = palabras[i:i + largo]
        if fuzz.ratio("".join(p.group() for p in ventana).lower(), objetivo) >= similitud:
            tramos.append((ventana[0].start(), ventana[-1].end()))
            i += largo
        else:
            i += 1
    for a, b in reversed(tramos):
        texto = texto[:a] + seudonimo + texto[b:]
    return texto, len(tramos)


def proteger(texto, corpus):
    for nombre, seudonimo in REEMPLAZOS.items():
        texto, n = reemplazar_nombre(texto, nombre, seudonimo)
        reemplazos_hechos[corpus][seudonimo] += n
    for etiqueta, patron in PATRONES.items():
        texto, n = re.subn(patron, etiqueta, texto)
        reemplazos_hechos[corpus][etiqueta] += n
    return texto


catalogo_docs["texto_ocr"] = catalogo_docs.texto_crudo.fillna("").map(lambda t: proteger(t, "documentos"))
segmentos = {c: [s | {"texto": proteger(s["texto"], "audio")} for s in segs] for c, segs in segmentos_crudos.items()}

for c, segs in segmentos.items():                 # exportación por clip: CSV, SRT y TXT
    base = DIR_AUD / "salidas" / c
    pd.DataFrame(segs, columns=["inicio", "fin", "hablante", "texto"]).to_csv(f"{base}.csv", index=False)
    Path(f"{base}.srt").write_text(a_srt(segs), encoding="utf-8")
    Path(f"{base}.txt").write_text(a_txt(segs), encoding="utf-8")


def unir_con_marcas(segs):
    "'[HABLANTE_1] ... [HABLANTE_2] ...'; sin marcas cuando no hubo separación de hablantes."
    return " ".join(f"[HABLANTE_{h[1:]}] {texto}" if h else texto for _, h, texto in turnos(segs))


catalogo_aud["texto_asr"] = catalogo_aud.id.map(lambda c: unir_con_marcas(segmentos[c]))
catalogo_aud["n_hablantes"] = catalogo_aud.id.map(lambda c: len({s["hablante"] for s in segmentos[c] if s["hablante"]}) or 1)

tabla_seud = pd.DataFrame(reemplazos_hechos).fillna(0).astype(int)
tabla_seud = tabla_seud[tabla_seud.sum(axis=1) > 0]
print(f"Entidades reemplazadas: {tabla_seud.values.sum()} "
      f"(documentos: {tabla_seud['documentos'].sum()}, audio: {tabla_seud['audio'].sum()})")
tabla_seud

Entidades reemplazadas: 18 (documentos: 11, audio: 7)


,documentos,audio
Persona_01,0,4
Persona_02,0,3
Persona_03,2,0
Persona_04,4,0
Persona_05,1,0
Persona_06,1,0
Persona_07,1,0
Persona_08,1,0
Persona_09,1,0


## Paso 5 · Limpieza

Reglas de descarte:

| Corpus | Motivo | Criterio |
|---|---|---|
| Ambos | Duplicado | Misma huella SHA-256 que otro archivo |
| Documentos | Ilegible | El OCR no encontró texto (< 10 caracteres) o la confianza mediana es < 0,50 |
| Audio | Sin voz | El VAD detecta menos de 1 s de voz o la transcripción sale vacía |

Un manuscrito difícil suele tener confianza mediana alrededor de 0,6 y no se descarta: solo se elimina lo que el modelo no puede leer en absoluto. De los documentos válidos tomo los primeros 8 impresos, 8 fotos y 9 manuscritos del catálogo; el resto queda de reserva.

In [35]:
from PIL import ImageFilter, ImageOps

# Segundo intento: ampliar 2x, contraste y enfoque, para documentos sin texto detectado
sin_texto = catalogo_docs[(catalogo_docs.error == "") & (catalogo_docs.n_lineas == 0)]
for doc in sin_texto.itertuples():
    gris = Image.open(doc.archivo).convert("L")
    print(f"{doc.id}: tamaño original {gris.size}")
    grande = gris.resize((gris.width * 2, gris.height * 2), Image.LANCZOS)
    mejorada = ImageOps.autocontrast(grande, cutoff=2).filter(ImageFilter.SHARPEN).convert("RGB")
    ruta = DIR_DOC / "ocr" / doc.id / "preprocesada.png"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    mejorada.save(ruta)
    lineas[doc.id] = extraer_texto(doc._replace(archivo=ruta))
    catalogo_docs.loc[catalogo_docs.id == doc.id, "modelo_ocr"] = (
        DESCRIPCION_OCR[doc.tipo] + " + preprocesado: escala de grises, ampliación 2x, autocontraste y enfoque")
    print(f"{doc.id}: {len(lineas[doc.id])} líneas tras el preprocesamiento")

catalogo_docs["n_lineas"] = catalogo_docs.id.map(lambda i: len(lineas.get(i, [])))
catalogo_docs["conf_mediana"] = catalogo_docs.id.map(
    lambda i: lineas[i].confianza.median() if len(lineas.get(i, [])) else np.nan)
catalogo_docs["texto_crudo"] = catalogo_docs.id.map(lambda i: "\n".join(lineas[i].texto) if i in lineas else "")

doc_024: tamaño original (362, 550)
doc_024: 2 líneas tras el preprocesamiento


In [32]:
print(lineas["doc_024"])
print("Caracteres:", len(catalogo_docs.loc[catalogo_docs.id == "doc_024", "texto_crudo"].iloc[0]))

            texto  confianza
0  edoed condiedn      0.428
1           t  le      0.337
Caracteres: 20


In [39]:
def motivo_documento(d):
    if d.error:
        return f"no se pudo obtener: {d.error}"
    if d.duplicado:
        return "duplicado"
    if d.n_lineas == 0 or len(d.texto_ocr.strip()) < 10:
        return "ilegible: sin texto detectado"
    if d.conf_mediana < 0.50:
        return "ilegible: confianza mediana < 0,50"
    return ""


def motivo_audio(a):
    if a.duplicado:
        return "duplicado"
    if a.segundos_voz < 1.0 or not a.texto_asr.strip():
        return "sin voz"
    return ""


catalogo_docs["duplicado"] = catalogo_docs.hash.notna() & catalogo_docs.duplicated("hash")
catalogo_docs["motivo_descarte"] = [motivo_documento(d) for d in catalogo_docs.itertuples()]
catalogo_aud["duplicado"] = catalogo_aud.duplicated("hash")
catalogo_aud["motivo_descarte"] = [motivo_audio(a) for a in catalogo_aud.itertuples()]

CUOTAS = {"impreso": 8, "foto": 8, "manuscrito": 9}
aptos = catalogo_docs[catalogo_docs.motivo_descarte == ""]
docs = pd.concat([aptos[aptos.tipo == t].head(n) for t, n in CUOTAS.items()]).copy()
clips = catalogo_aud[catalogo_aud.motivo_descarte == ""].copy()

descartes = pd.concat([
    catalogo_docs.loc[catalogo_docs.motivo_descarte != "", ["id", "tipo", "origen", "motivo_descarte"]]
        .assign(corpus="documentos"),
    catalogo_aud.loc[catalogo_aud.motivo_descarte != "", ["id", "descripcion", "motivo_descarte"]]
        .rename(columns={"descripcion": "origen"}).assign(corpus="audio", tipo="audio"),
], ignore_index=True)
descartes.to_csv(DATOS / "descartes.csv", index=False)

print("Descartes:", "ninguno" if descartes.empty else "")
if not descartes.empty:
    print(descartes.groupby(["corpus", "motivo_descarte"]).size().to_string())
print(f"Documentos aptos de reserva (no usados): {len(aptos) - len(docs)}")
faltantes = {t: n - (docs.tipo == t).sum() for t, n in CUOTAS.items() if (docs.tipo == t).sum() < n}
if faltantes:
    print(f"⚠ Faltan documentos aptos {faltantes}: agrega más a CANDIDATOS y vuelve a ejecutar.")
minutos_audio = clips.duracion_seg.sum() / 60
print(f"Corpus final: {len(docs)} documentos {docs.tipo.value_counts().to_dict()} · {len(clips)} clips, "
      f"{minutos_audio:.1f} min, {(clips.n_hablantes >= 2).sum()} con 2+ hablantes")
assert not faltantes and len(docs) >= 25, "No se alcanza el mínimo de documentos"
assert minutos_audio >= 30 and len(clips) >= 20 and (clips.n_hablantes >= 2).any(), "No se alcanza el mínimo de audio"
descartes

Descartes: 
corpus      motivo_descarte                   
documentos  ilegible: confianza mediana < 0,50    1
Documentos aptos de reserva (no usados): 4
Corpus final: 25 documentos {'manuscrito': 9, 'impreso': 8, 'foto': 8} · 22 clips, 32.2 min, 2 con 2+ hablantes


,id,tipo,origen,motivo_descarte,corpus
0,doc_024,manuscrito,Capitulacón de Ayacucho2.jpg,"ilegible: confianza mediana < 0,50",documentos


## Paso 6 · Exploración

Los títulos de los gráficos se generan a partir de los datos, de modo que cada uno enuncia lo que muestra.

In [40]:
docs["n_caracteres"] = docs.texto_ocr.str.len()
clips["n_palabras"] = clips.texto_asr.str.replace(r"\[HABLANTE_\d+\]", "", regex=True).str.split().str.len()
clips["tipo_audio"] = np.select(
    [clips.fuente.str.contains("diálogo"), clips.fuente.str.startswith("Wikimedia")],
    ["Diálogo (OpenSLR 73)", "Entrevista (Commons)"], "Lectura (OpenSLR 73)")

resumen_tipo = (docs.groupby("tipo").agg(documentos=("id", "size"), mediana_caracteres=("n_caracteres", "median"))
                .reindex(list(CUOTAS)).reset_index())
mediana = resumen_tipo.set_index("tipo").mediana_caracteres
razon = mediana["impreso"] / max(mediana["manuscrito"], 1)
frase = f"{razon:.1f} veces más" if razon >= 1 else f"{1 / razon:.1f} veces menos"
titulo = (f"Corpus balanceado ({'/'.join(map(str, resumen_tipo.documentos))}), pero un impreso entrega "
          f"{frase} texto que un manuscrito (mediana de caracteres)")

grafico = px.bar(resumen_tipo, x="tipo", y="documentos", color="tipo", text="documentos",
                 hover_data={"mediana_caracteres": ":.0f"}, title=titulo,
                 labels={"tipo": "Tipo de documento", "documentos": "N.º de documentos",
                         "mediana_caracteres": "Mediana de caracteres"})
grafico.update_layout(showlegend=False)
grafico.show()

In [41]:
minutos_por_tipo = clips.groupby("tipo_audio").duracion_seg.sum().div(60)
principal = minutos_por_tipo.idxmax()
titulo = (f"{principal} aporta el {minutos_por_tipo.max() / minutos_por_tipo.sum():.0%} de los "
          f"{minutos_por_tipo.sum():.0f} minutos; los clips duran entre {clips.duracion_seg.min():.0f} s "
          f"y {clips.duracion_seg.max() / 60:.1f} min")

grafico = px.bar(clips.sort_values("duracion_seg", ascending=False), x="id", y="duracion_seg", color="tipo_audio",
                 hover_data=["descripcion", "n_hablantes"], title=titulo,
                 labels={"id": "Clip", "duracion_seg": "Duración (s)", "tipo_audio": "Tipo de audio"})
grafico.update_xaxes(categoryorder="total descending")
grafico.show()

In [42]:
longitudes = pd.concat([
    pd.DataFrame({"medida": "Documentos · caracteres OCR", "longitud": docs.n_caracteres, "id": docs.id}),
    pd.DataFrame({"medida": "Audio · palabras ASR", "longitud": clips.n_palabras, "id": clips.id}),
])
cola_doc = docs.n_caracteres.max() / max(docs.n_caracteres.median(), 1)
cola_aud = clips.n_palabras.max() / max(clips.n_palabras.median(), 1)
titulo = (f"Distribuciones con cola a la derecha: el texto OCR más largo mide {cola_doc:.0f}× la mediana "
          f"y la transcripción más larga {cola_aud:.0f}× la mediana")

grafico = px.histogram(longitudes, x="longitud", facet_col="medida", nbins=15, hover_data=["id"], title=titulo,
                       labels={"longitud": "Longitud"})
grafico.update_xaxes(matches=None, showticklabels=True)
grafico.update_yaxes(title_text="Frecuencia", col=1)
grafico.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
grafico.show()

## Paso 7 · Validación

Reviso los extremos: deben aparecer en las colas de los histogramas y su texto debe ser coherente. Un documento «larguísimo» lleno de líneas repetidas o un clip corto sin palabras indicaría un problema de extracción.

In [43]:
columnas = ["id", "tipo", "n_caracteres", "texto_ocr"]
print("Documentos con más texto OCR");  display(docs.nlargest(3, "n_caracteres")[columnas])
print("Documentos con menos texto OCR"); display(docs.nsmallest(3, "n_caracteres")[columnas])

columnas = ["id", "tipo_audio", "duracion_seg", "n_palabras", "n_hablantes", "texto_asr"]
print("Clips más largos"); display(clips.nlargest(3, "duracion_seg")[columnas])
print("Clips más cortos"); display(clips.nsmallest(3, "duracion_seg")[columnas])

for nombre, serie in [("caracteres OCR", docs.n_caracteres), ("palabras ASR", clips.n_palabras)]:
    q1, q3 = serie.quantile([0.25, 0.75])
    print(f"{nombre}: mín {serie.min()} · Q1 {q1:.0f} · mediana {serie.median():.0f} · Q3 {q3:.0f} · máx {serie.max()}")
print(f"Correlación duración–palabras: {clips[['duracion_seg', 'n_palabras']].corr().iloc[0, 1]:.2f}")

Documentos con más texto OCR


,id,tipo,n_caracteres,texto_ocr
11,doc_012,impreso,9176,Bste periódieo se despacha en esta Capital\nLos avisos que deban samr en este\nen a ti...
10,doc_011,impreso,5117,EL COMERCIO\n826\nras siendo el numero ultimo 183.\nSe está esperando un decreto del G...
12,doc_013,impreso,2990,"CEDVLA REAL DEL\nSEÑOR\nEMPERADOR\nCARLOS V. Y LA REYNA DOÁA JUANA SU MA-\ndre, fobre ..."


Documentos con menos texto OCR


,id,tipo,n_caracteres,texto_ocr
6,doc_007,foto,15,Espaderos\nCalle
2,doc_003,foto,30,NE COLON\nBOLETERIA\nPLATEA ALTA
7,doc_008,foto,36,Plazoleta Santa Catalina\nia peatonal


Clips más largos


,id,tipo_audio,duracion_seg,n_palabras,n_hablantes,texto_asr
0,clip_001,Entrevista (Commons),196.0,518,1,"[HABLANTE_1] A ver, a ver, ¿cuál es la pregunta? Buenas noches, primeramente. Mi nombr..."
3,clip_004,Entrevista (Commons),163.9,325,1,"[HABLANTE_1] Buenas noches, soy Jean-Persona_01, gestionante de la San Marcos. ¿Cuál e..."
1,clip_002,Entrevista (Commons),96.1,181,1,"[HABLANTE_1] Nos encontramos aquí con el señor Carlos Ramírez, señor Carlos, esta entr..."


Clips más cortos


,id,tipo_audio,duracion_seg,n_palabras,n_hablantes,texto_asr
4,clip_005,Entrevista (Commons),54.3,159,1,"[HABLANTE_1] Sí, déjalo ya, así, ya está grabando, ya está grabando, ya está grabando ..."
10,clip_011,Lectura (OpenSLR 73),76.3,104,1,Un poco de humor nunca cae mal después de estos desastres Quiero transferir dinero vía...
9,clip_010,Lectura (OpenSLR 73),76.4,110,1,Mañana juegan los cuatro equipos más importantes de la liga estatal La ciudad es bellí...


caracteres OCR: mín 15 · Q1 117 · mediana 451 · Q3 775 · máx 9176
palabras ASR: mín 79 · Q1 107 · mediana 118 · Q3 159 · máx 518
Correlación duración–palabras: 0.92


## Paso 8 · Exportación

| Archivo | Contenido |
|---|---|
| `datos/documentos.csv` | `id`, `archivo`, `tipo`, `fuente`, `texto_ocr`, `modelo_ocr`, `texto_referencia` |
| `datos/transcripciones.csv` | `id`, `archivo_audio`, `duracion_seg`, `idioma`, `texto_asr`, `modelo_asr`, `n_hablantes`, `texto_referencia` |
| `datos/fuentes.csv` | `id`, `modalidad`, `fuente_url`, `licencia`, `acceso` |
| `datos/atribucion.csv` | autor y licencia de cada archivo |
| `datos/descartes.csv` | archivos descartados y motivo |
| `datos/raw/audio/referencias_fuente.csv` | transcripciones oficiales de OpenSLR 73 para la Parte 2 (no se versiona) |
| `datos/raw/audio/originales/dialogo_*_guion.csv` | quién dice qué y cuándo en cada diálogo |

`texto_referencia` queda vacío por ahora y se completa en la Parte 2 con la muestra de evaluación.

In [44]:
documentos_csv = docs.assign(archivo=docs.archivo.map(ruta_relativa), texto_referencia="")[
    ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia"]]
transcripciones_csv = clips.assign(archivo_audio=clips.archivo_audio.map(ruta_relativa), idioma="es",
                                   texto_referencia="")[
    ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes", "texto_referencia"]]
fuentes_csv = pd.concat([
    docs.assign(modalidad="documento")[["id", "modalidad", "fuente_url", "licencia", "acceso"]],
    clips.assign(modalidad="audio")[["id", "modalidad", "fuente_url", "licencia", "acceso"]],
], ignore_index=True)
atribucion_csv = pd.concat([
    docs.assign(archivo=docs.archivo.map(ruta_relativa))[["id", "archivo", "autor", "licencia", "fuente_url"]],
    clips.assign(archivo=clips.archivo_audio.map(ruta_relativa))[["id", "archivo", "autor", "licencia", "fuente_url"]],
], ignore_index=True)

documentos_csv.to_csv(DATOS / "documentos.csv", index=False)
transcripciones_csv.to_csv(DATOS / "transcripciones.csv", index=False)
fuentes_csv.to_csv(DATOS / "fuentes.csv", index=False)
atribucion_csv.to_csv(DATOS / "atribucion.csv", index=False)
clips.loc[clips.referencia_fuente != "", ["id", "fuente", "referencia_fuente"]].to_csv(
    DIR_AUD / "referencias_fuente.csv", index=False)

for nombre, tabla in [("documentos", documentos_csv), ("transcripciones", transcripciones_csv), ("fuentes", fuentes_csv)]:
    print(f"datos/{nombre}.csv → {tabla.shape}")
documentos_csv.head()

datos/documentos.csv → (25, 7)
datos/transcripciones.csv → (22, 8)
datos/fuentes.csv → (47, 5)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia
10,doc_011,datos/raw/documentos/doc_011.jpg,impreso,Wikimedia Commons (Public domain),EL COMERCIO\n826\nras siendo el numero ultimo 183.\nSe está esperando un decreto del G...,"PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientac...",
11,doc_012,datos/raw/documentos/doc_012.jpg,impreso,Wikimedia Commons (CC BY-SA 4.0),Bste periódieo se despacha en esta Capital\nLos avisos que deban samr en este\nen a ti...,"PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientac...",
12,doc_013,datos/raw/documentos/doc_013.png,impreso,Wikimedia Commons (Public domain),"CEDVLA REAL DEL\nSEÑOR\nEMPERADOR\nCARLOS V. Y LA REYNA DOÁA JUANA SU MA-\ndre, fobre ...","PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientac...",
13,doc_014,datos/raw/documentos/doc_014.jpg,impreso,Wikimedia Commons (Public domain),"CREAM CRACKER""\nEl Comercio\nDr. W. J. Morgan Optumétrico\n“FIELD”\nEgecialsado\n137 P...","PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientac...",
14,doc_015,datos/raw/documentos/doc_015.jpg,impreso,Wikimedia Commons (Public domain),A+GINCO+CIN\n8\n18\nJouna\nJo\nICI\nELBANCONACIONAL-PERU\naqard CINCO SOLES ala visla\...,"PaddleOCR 3.7.0 PP-OCRv6 medium (lang=es, engine=transformers, cpu, ≤1600 px, orientac...",


In [45]:
transcripciones_csv.head()

,id,archivo_audio,duracion_seg,idioma,texto_asr,modelo_asr,n_hablantes,texto_referencia
0,clip_001,datos/raw/audio/clip_001.wav,196.0,es,"[HABLANTE_1] A ver, a ver, ¿cuál es la pregunta? Buenas noches, primeramente. Mi nombr...","faster-whisper 1.2.1 large-v3-turbo (int8, beam 5, VAD on, idioma=es) + separación de ...",1,
1,clip_002,datos/raw/audio/clip_002.wav,96.1,es,"[HABLANTE_1] Nos encontramos aquí con el señor Carlos Ramírez, señor Carlos, esta entr...","faster-whisper 1.2.1 large-v3-turbo (int8, beam 5, VAD on, idioma=es) + separación de ...",1,
2,clip_003,datos/raw/audio/clip_003.wav,77.4,es,"[HABLANTE_1] Buenas noches, soy Jean-Persona_01, estudiante de la San Marcos, esta ent...","faster-whisper 1.2.1 large-v3-turbo (int8, beam 5, VAD on, idioma=es) + separación de ...",1,
3,clip_004,datos/raw/audio/clip_004.wav,163.9,es,"[HABLANTE_1] Buenas noches, soy Jean-Persona_01, gestionante de la San Marcos. ¿Cuál e...","faster-whisper 1.2.1 large-v3-turbo (int8, beam 5, VAD on, idioma=es) + separación de ...",1,
4,clip_005,datos/raw/audio/clip_005.wav,54.3,es,"[HABLANTE_1] Sí, déjalo ya, así, ya está grabando, ya está grabando, ya está grabando ...","faster-whisper 1.2.1 large-v3-turbo (int8, beam 5, VAD on, idioma=es) + separación de ...",1,


In [46]:
fuentes_csv.head()

,id,modalidad,fuente_url,licencia,acceso
0,doc_011,documento,https://commons.wikimedia.org/wiki/File:El_Comercio_-_Lima._Per%C3%BA_y_Ecuador._3-3-1...,Public domain,descarga pública de Wikimedia Commons (ancho máx. 1600 px)
1,doc_012,documento,https://commons.wikimedia.org/wiki/File:El_Comercio_Issue_12.jpg,CC BY-SA 4.0,descarga pública de Wikimedia Commons (ancho máx. 1600 px)
2,doc_013,documento,https://commons.wikimedia.org/wiki/File:Cedula_Real_1551_Universidad_de_San_Marcos.png,Public domain,descarga pública de Wikimedia Commons (ancho máx. 1600 px)
3,doc_014,documento,"https://commons.wikimedia.org/wiki/File:El_Comercio,_24_de_mayo_de_1940.jpg",Public domain,descarga pública de Wikimedia Commons (ancho máx. 1600 px)
4,doc_015,documento,https://commons.wikimedia.org/wiki/File:Billete_de_1870_de_5_soles_del_Banco_Nacional_...,Public domain,descarga pública de Wikimedia Commons (ancho máx. 1600 px)


## Conclusiones

- **Documentos:** de 30 candidatos peruanos de Wikimedia Commons con licencia libre se obtuvo un corpus de **25 documentos** (8 impresos, 8 fotos y 9 manuscritos). PaddleOCR leyó bien los impresos (confianza mediana de 0,72 a 1,00) y de forma aceptable los manuscritos (0,52 a 0,94). Se descartó **1 documento** (doc_024, *Capitulación de Ayacucho*) por ilegible, incluso después de preprocesarlo con escala de grises, ampliación, autocontraste y enfoque.
- **Audio:** **22 clips y 32,2 minutos** (5 entrevistas de Commons, 2 diálogos y 15 lecturas de OpenSLR 73), transcritos con faster-whisper `large-v3-turbo` en 7,2 minutos en CPU. La separación de hablantes detectó correctamente **2 voces en los 2 diálogos** de verdad conocida; en las entrevistas detectó una sola, lo que es una limitación del modo automático.
- **Seudonimización:** se reemplazaron **18 entidades** (11 en documentos y 7 en audio) correspondientes a **9 personas**, incluidas personas históricas, por criterio conservador. No se encontraron DNI, teléfonos ni correos.
- **Validación:** los textos OCR van de 15 caracteres (letreros) a 9 176 (periódico de 1842), con mediana de 451, sesgados por pocos documentos densos. En audio, la correlación entre duración y palabras transcritas es **0,92**, lo que confirma un ASR consistente.